# Evaluate Models (v4)
One evaluator for all 23 models (W6): mAP50 / mAP50-95 / per-class AP / small-medium-large breakdown / confusion matrix / speed. Results land in each run dir as `eval_<dataset>_<split>.json`.


In [ ]:
# v4 bootstrap: mount Drive, install pinned deps, verify cleaned data
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

REPO = Path("/content/minor-project") if IN_COLAB else Path.cwd()
if not (REPO / "training" / "run.py").exists():
    # adjust to your Drive copy of the repo
    REPO = Path("/content/drive/MyDrive/object-detection/minor-project")
os.chdir(REPO)
print("repo:", REPO)

# single installer entry (idempotent, W8.1)
subprocess.check_call([sys.executable, "install.py"])

# environment report - paste this into any bug report (W8.5)
subprocess.check_call([sys.executable, "training/run.py", "--env"])

# verify cleaned data; if missing, run the cleaner first:
#   python data_cleaning/clean_build_v4.py --help
DATA = Path("/content/drive/MyDrive/object-detection/cleaned_data") if IN_COLAB else REPO / "data" / "cleaned_data"
for ds in ("codrone", "visdrone", "dota"):
    ok = (DATA / ds / "train" / "images").is_dir()
    print(f"{ds:<10}: {'ready' if ok else 'MISSING - run data_cleaning/clean_build_v4.py'}")


## Single run


In [ ]:
# Evaluate one run (one protocol for all 23 models, W6)
!python training/evaluate.py --run runs/train/visdrone_yolo26s-obb --datasets visdrone


## Cross-dataset matrix + heatmap


In [ ]:
# Cross-dataset matrix (stage once, reuse for every cell, W6.2)
!python training/cross_eval.py --runs-dir runs/train --stage


## Master summary table (dataset x model -> mAP / params / latency)


In [ ]:
!python training/summarize_runs.py --runs-dir runs --out results/master_summary.csv
